# Ecos Progressivos — End-to-End Marketing Performance Analytics

## 1. Contexto do Projeto

Este projeto analisa o desempenho de uma campanha digital criada para captar subscritores para a newsletter Ecos Progressivos, dedicada ao rock progressivo.

O pipeline integra dados provenientes de Meta Ads, Google Analytics 4 e Mailchimp, que são tratados em Python, armazenados em SQL Server e posteriormente analisados no Power BI.

## 2. Objetivos de Negócio

A análise procura responder às seguintes questões:

1. Qual foi o alcance e o investimento da campanha Meta Ads?
2. Quantas sessões e novos utilizadores foram registados no website?
3. Quantos inícios de formulário foram registados?
4. Quantas subscrições foram confirmadas no Mailchimp?
5. Em que fase do funil ocorreu a maior perda?
6. Qual foi o custo associado à aquisição de subscritores?
7. Que melhorias devem ser testadas em campanhas futuras?

## 3. Importação de Bibliotecas e Configuração



In [24]:
from pathlib import Path
import pandas as pd

In [25]:
# Diretório principal do projeto
BASE_DIR = Path.cwd()

# Ficheiros de origem
META_FILE = (
    BASE_DIR
    / "Conta-Ecos-Progressivos-Campanhas-01_11_2025-30_11_2025.csv"
)

GA4_FILE = (
    BASE_DIR
    / "Resumo_dos_relatórios.csv"
)

MAILCHIMP_FILE = (
    BASE_DIR
    / "subscribed_email_audience_export_fc258fbd07.csv"
)

print(f"Diretório do projeto: {BASE_DIR}")

Diretório do projeto: /content


In [26]:
ficheiros_origem = {
    "Meta Ads": META_FILE,
    "Google Analytics 4": GA4_FILE,
    "Mailchimp": MAILCHIMP_FILE
}

for plataforma, caminho in ficheiros_origem.items():
    if caminho.exists():
        print(f"✓ {plataforma}: ficheiro encontrado")
    else:
        print(f"✗ {plataforma}: ficheiro não encontrado")
        print(f"  Caminho procurado: {caminho}")

✓ Meta Ads: ficheiro encontrado
✓ Google Analytics 4: ficheiro encontrado
✓ Mailchimp: ficheiro encontrado


## 4. Importação e Análise dos Dados da Meta Ads

Nesta etapa é importado o relatório da campanha Meta Ads relativo ao período de 1 a 30 de novembro de 2025.

O objetivo consiste em validar a estrutura do ficheiro, identificar a campanha analisada e extrair as principais métricas de desempenho publicitário: investimento, impressões, alcance, resultados atribuídos e custo por resultado.



In [27]:
df_meta = pd.read_csv(META_FILE)

In [28]:
df_meta.head()

,Início dos relatórios,Fim dos relatórios,Nome da campanha,Apresentação de campanhas,Resultados,Indicador de resultados,Custo por resultados,Orçamento dos conjuntos de anúncios,Tipo de orçamento do conjunto de anúncios,Montante gasto (EUR),Impressões,Alcance,Termina a,Definição de atribuição,Resultados (iniciais),Indicador de resultados (inicial)
0,2025-11-01,2025-11-30,Ecos Progressivos - conversões nov25,completed,7,actions:offsite_conversion.custom.137071381133...,9.255714,65,Toda a duração,64.79,25386,16338,2025-11-25,"Clique com 7 dias, visualização com 1 dia ou v...",NaN,NaN


In [29]:
# Campanha analisada
NOME_CAMPANHA = "Ecos Progressivos - conversões nov25"

# Seleção dos registos correspondentes à campanha
df_campanha_meta = df_meta.loc[
    df_meta["Nome da campanha"]
    .astype(str)
    .str.strip()
    .eq(NOME_CAMPANHA)
].copy()

if df_campanha_meta.empty:
    raise ValueError(
        f"A campanha '{NOME_CAMPANHA}' não foi encontrada "
        "no relatório da Meta Ads."
    )

print(
    f"✓ Campanha encontrada: {NOME_CAMPANHA}"
)

print(
    f"Número de registos da campanha: "
    f"{len(df_campanha_meta)}"
)

✓ Campanha encontrada: Ecos Progressivos - conversões nov25
Número de registos da campanha: 1


In [30]:
# Conversão das datas do relatório
periodo_inicio = pd.to_datetime(
    df_campanha_meta["Início dos relatórios"],
    errors="coerce"
).min()

periodo_fim = pd.to_datetime(
    df_campanha_meta["Fim dos relatórios"],
    errors="coerce"
).max()

# Métricas principais da Meta Ads
investimento_meta = float(
    df_campanha_meta["Montante gasto (EUR)"].sum()
)

impressoes_meta = int(
    df_campanha_meta["Impressões"].sum()
)

alcance_meta = int(
    df_campanha_meta["Alcance"].sum()
)

resultados_meta = int(
    df_campanha_meta["Resultados"].sum()
)

# Cálculo independente do custo por resultado
if resultados_meta > 0:
    custo_por_resultado_meta = (
        investimento_meta / resultados_meta
    )
else:
    custo_por_resultado_meta = None

# Definição de atribuição utilizada pela Meta
definicao_atribuicao_meta = (
    df_campanha_meta["Definição de atribuição"]
    .dropna()
    .iloc[0]
)

In [31]:
print(
    "Período da campanha:",
    periodo_inicio.strftime("%d/%m/%Y"),
    "a",
    periodo_fim.strftime("%d/%m/%Y")
)

print(
    "\nDefinição de atribuição da Meta:"
)

print(definicao_atribuicao_meta)

Período da campanha: 01/11/2025 a 30/11/2025

Definição de atribuição da Meta:
Clique com 7 dias, visualização com 1 dia ou visualização com interação de 1 dia


### Interpretação das Métricas da Meta Ads

A campanha registou um investimento de **64,79 €**, com **25 386 impressões** e um alcance de **16 338 pessoas**.

A plataforma atribuiu **7 resultados** à campanha, o que corresponde a um custo de aproximadamente **9,26 € por resultado atribuído**.

Os resultados atribuídos pela Meta não devem ser automaticamente equiparados ao número total de subscrições confirmadas no Mailchimp. As duas plataformas utilizam métodos de medição e atribuição distintos, pelo que os respetivos indicadores serão analisados separadamente.

## 5. Importação e Análise dos Dados do Mailchimp

O relatório do Mailchimp é utilizado para quantificar as subscrições registadas durante o período de análise.

Como o ficheiro contém dados pessoais, incluindo endereços de email e endereços IP, a análise é efetuada sem apresentar registos individuais. Apenas são extraídas métricas agregadas necessárias para o projeto.




In [32]:
# Importação do relatório do Mailchimp
df_mailchimp = pd.read_csv(
    MAILCHIMP_FILE,
    encoding="utf-8-sig"
)

print(
    f"Número de registos importados: "
    f"{len(df_mailchimp)}"
)

print(
    "✓ Ficheiro do Mailchimp importado "
    "sem apresentar dados pessoais."
)

Número de registos importados: 16
✓ Ficheiro do Mailchimp importado sem apresentar dados pessoais.


### Cálculo das Subscrições Confirmadas

Para proteger os dados pessoais, o notebook não apresenta os registos individuais. Os endereços de email são utilizados apenas para contar contactos únicos com uma data de confirmação válida.

In [33]:
# Normalização dos endereços de email
emails_mailchimp = (
    df_mailchimp["Endereço de e-mail"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Conversão das datas de confirmação
datas_confirmacao_mailchimp = pd.to_datetime(
    df_mailchimp["CONFIRM_TIME"],
    errors="coerce"
)

# Contagem de emails únicos com confirmação válida
subscricoes_confirmadas = int(
    emails_mailchimp[
        datas_confirmacao_mailchimp.notna()
    ]
    .dropna()
    .nunique()
)

# Período das subscrições confirmadas
periodo_mailchimp_inicio = (
    datas_confirmacao_mailchimp.min()
)

periodo_mailchimp_fim = (
    datas_confirmacao_mailchimp.max()
)

print(
    f"Subscrições confirmadas: "
    f"{subscricoes_confirmadas}"
)

print(
    "Período das subscrições:",
    periodo_mailchimp_inicio.strftime(
        "%d/%m/%Y"
    ),
    "a",
    periodo_mailchimp_fim.strftime(
        "%d/%m/%Y"
    )
)

Subscrições confirmadas: 16
Período das subscrições: 12/11/2025 a 24/11/2025


### Interpretação

O Mailchimp registou **16 subscrições confirmadas** durante o período analisado.

Os registos individuais não são apresentados, uma vez que o ficheiro contém dados pessoais. O Mailchimp é utilizado como fonte de validação das subscrições, mas a exportação não permite atribuir cada contacto diretamente a uma campanha ou canal.

## 6. Importação e Análise dos Dados do GA4

A exportação do Google Analytics 4 contém várias tabelas agregadas no mesmo ficheiro CSV.

Para esta análise foi selecionada a secção correspondente à contagem de eventos, que inclui as principais métricas necessárias para a construção do funil:

- sessões;
- novos utilizadores;
- eventos de scroll;
- inícios de formulário.

Como a tabela é selecionada através da sua posição no ficheiro, uma nova exportação do GA4 pode exigir o ajuste do parâmetro `skiprows`.

In [34]:
df_ga4_eventos = pd.read_csv(
    GA4_FILE,
    skiprows=243,
    nrows=6
)

display(df_ga4_eventos)

,Nome do evento,Contagem de eventos
0,page_view,614
1,session_start,532
2,first_visit,481
3,scroll,122
4,user_engagement,111
5,form_start,43


In [35]:
df_ga4_eventos[
    "Contagem de eventos"
] = pd.to_numeric(
    df_ga4_eventos["Contagem de eventos"],
    errors="coerce"
)

In [36]:
sessoes_ga4 = int(
    df_ga4_eventos.loc[
        df_ga4_eventos["Nome do evento"] == "session_start",
        "Contagem de eventos"
    ].iloc[0]
)

novos_utilizadores_ga4 = int(
    df_ga4_eventos.loc[
        df_ga4_eventos["Nome do evento"] == "first_visit",
        "Contagem de eventos"
    ].iloc[0]
)

eventos_scroll_ga4 = int(
    df_ga4_eventos.loc[
        df_ga4_eventos["Nome do evento"] == "scroll",
        "Contagem de eventos"
    ].iloc[0]
)

inicios_formulario_ga4 = int(
    df_ga4_eventos.loc[
        df_ga4_eventos["Nome do evento"] == "form_start",
        "Contagem de eventos"
    ].iloc[0]
)

In [37]:
resumo_ga4 = pd.DataFrame({
    "Métrica": [
        "Sessões",
        "Novos utilizadores",
        "Eventos de scroll",
        "Inícios de formulário"
    ],
    "Valor": [
        sessoes_ga4,
        novos_utilizadores_ga4,
        eventos_scroll_ga4,
        inicios_formulario_ga4
    ]
})

display(resumo_ga4)

,Métrica,Valor
0,Sessões,532
1,Novos utilizadores,481
2,Eventos de scroll,122
3,Inícios de formulário,43


### Interpretação dos Dados do GA4

Durante o período analisado, o website registou **532 sessões** e **481 novos utilizadores**.

Foram também registados **122 eventos de scroll** e **43 inícios de formulário**.

Os 43 inícios de formulário correspondem à contagem do evento `form_start` no GA4. Este valor não representa necessariamente 43 utilizadores únicos, uma vez que o mesmo utilizador pode iniciar o formulário em sessões diferentes.

Para a construção do funil, as **532 sessões** são utilizadas como base de tráfego, substituindo a designação anterior de “481 visitas”. Os 481 registos de `first_visit` representam novos utilizadores e não o total de visitas ao website.

## 7. Construção do Funil e Cálculo dos KPIs

Nesta etapa são integradas as métricas agregadas da Meta Ads, do Google Analytics 4 e do Mailchimp para avaliar o desempenho da campanha ao longo do funil de aquisição.

O funil principal analisa a jornada dentro do website:

1. sessões registadas no GA4;
2. inícios de formulário;
3. subscrições confirmadas no Mailchimp.

As impressões e o alcance da Meta Ads são analisados separadamente como métricas de topo de funil, evitando diferenças de escala que dificultariam a leitura da conversão dentro da landing page.

Como os inícios de formulário correspondem a eventos do GA4 e as subscrições correspondem a contactos únicos no Mailchimp, o rácio entre estas duas etapas deve ser interpretado como um indicador operacional e não como uma taxa de utilizadores únicos rigorosamente acompanhados.

In [38]:
taxa_inicio_formulario = (
    inicios_formulario_ga4 / sessoes_ga4
)

racio_subscricoes_inicio = (
    subscricoes_confirmadas
    / inicios_formulario_ga4
)

taxa_conversao_sessao = (
    subscricoes_confirmadas / sessoes_ga4
)

custo_resultado_meta = (
    investimento_meta / resultados_meta
)

custo_global_subscricao = (
    investimento_meta / subscricoes_confirmadas
)

cpm_meta = (
    investimento_meta / impressoes_meta
) * 1000

frequencia_media = (
    impressoes_meta / alcance_meta
)

In [39]:
resumo_kpis = pd.DataFrame({
    "KPI": [
        "Taxa de início do formulário",
        "Rácio subscrições/inícios",
        "Taxa de conversão por sessão",
        "Custo por resultado Meta",
        "Custo global por subscrição",
        "CPM",
        "Frequência média"
    ],
    "Valor": [
        taxa_inicio_formulario,
        racio_subscricoes_inicio,
        taxa_conversao_sessao,
        custo_resultado_meta,
        custo_global_subscricao,
        cpm_meta,
        frequencia_media
    ]
})

display(resumo_kpis)

,KPI,Valor
0,Taxa de início do formulário,0.080827
1,Rácio subscrições/inícios,0.372093
2,Taxa de conversão por sessão,0.030075
3,Custo por resultado Meta,9.255714
4,Custo global por subscrição,4.049375
5,CPM,2.552194
6,Frequência média,1.553801


In [40]:
df_funil = pd.DataFrame({
    "Etapa": [
        "Sessões",
        "Inícios de formulário",
        "Subscrições confirmadas"
    ],
    "Valor": [
        sessoes_ga4,
        inicios_formulario_ga4,
        subscricoes_confirmadas
    ]
})

display(df_funil)

,Etapa,Valor
0,Sessões,532
1,Inícios de formulário,43
2,Subscrições confirmadas,16


### Interpretação do Funil e dos KPIs

A campanha registou **25 386 impressões**, alcançou **16 338 pessoas** e apresentou uma frequência média de **1,55 impressões por pessoa**. O investimento total foi de **64,79 €**, correspondente a um CPM de aproximadamente **2,55 €**.

No website foram registadas **532 sessões**, das quais resultaram **43 inícios de formulário**. A taxa de início do formulário foi, assim, de **8,08%**.

O Mailchimp confirmou **16 subscrições**, o que corresponde a uma taxa global de conversão de **3,01% por sessão**.

O rácio entre as 16 subscrições confirmadas e os 43 inícios de formulário foi de **37,21%**. Contudo, este indicador compara eventos do GA4 com contactos únicos do Mailchimp e deve ser interpretado como uma aproximação operacional.

A maior perda proporcional ocorreu entre as sessões e o início do formulário: **91,92% das sessões não originaram um evento `form_start`**. Este resultado sugere que a otimização deve abranger não apenas o formulário, mas também a proposta de valor, a visibilidade do CTA e o alinhamento entre a campanha e a landing page.

O custo por resultado atribuído pela Meta foi de **9,26 €**, enquanto o custo global por subscrição registada no Mailchimp foi de **4,05 €**. Estes indicadores não são diretamente equivalentes, porque utilizam sistemas de atribuição e denominadores diferentes.

## 8. Consolidação dos Dados

As métricas principais das três plataformas são consolidadas num único dataframe em formato vertical.

Cada linha representa uma métrica agregada para o período da campanha. Esta estrutura facilita o carregamento no SQL Server e a posterior utilização dos dados no Power BI.

In [41]:
df_final = pd.DataFrame({
    "data_registo": [
        periodo_fim
    ] * 9,

    "plataforma": [
        "Meta Ads",
        "Meta Ads",
        "Meta Ads",
        "Meta Ads",
        "GA4",
        "GA4",
        "GA4",
        "GA4",
        "Mailchimp"
    ],

    "metrica": [
        "Investimento",
        "Impressoes",
        "Alcance",
        "Resultados_Meta",
        "Sessoes",
        "Novos_Utilizadores",
        "Eventos_Scroll",
        "Inicios_Formulario",
        "Subscricoes_Confirmadas"
    ],

    "valor": [
        investimento_meta,
        impressoes_meta,
        alcance_meta,
        resultados_meta,
        sessoes_ga4,
        novos_utilizadores_ga4,
        eventos_scroll_ga4,
        inicios_formulario_ga4,
        subscricoes_confirmadas
    ]
})

display(df_final)

,data_registo,plataforma,metrica,valor
0,2025-11-30,Meta Ads,Investimento,64.79
1,2025-11-30,Meta Ads,Impressoes,25386.00
2,2025-11-30,Meta Ads,Alcance,16338.00
3,2025-11-30,Meta Ads,Resultados_Meta,7.00
4,2025-11-30,GA4,Sessoes,532.00
5,2025-11-30,GA4,Novos_Utilizadores,481.00
6,2025-11-30,GA4,Eventos_Scroll,122.00
7,2025-11-30,GA4,Inicios_Formulario,43.00
8,2025-11-30,Mailchimp,Subscricoes_Confirmadas,16.00


Cada linha representa uma métrica agregada para o período da campanha. A coluna `data_registo` corresponde à data final do período analisado.

## 9. Carregamento no SQL Server

Após o tratamento e a consolidação dos dados em Python, o dataframe final é carregado no SQL Server.

O SQL Server funciona como camada de armazenamento entre o processo de ETL realizado em Python e a visualização desenvolvida no Power BI.

Durante o desenvolvimento, a tabela é substituída em cada execução para evitar a acumulação de registos duplicados. Num ambiente de produção, seria necessário implementar um processo incremental de atualização.

In [42]:
# Esta etapa só deve ser executada no computador
# onde está instalado o SQL Server

EXECUTAR_SQL = False

if EXECUTAR_SQL:
    import urllib.parse
    from sqlalchemy import create_engine

    servidor = r"DESKTOP-9RHK5JJ\SQLEXPRESS"
    base_dados = "EcosProgressivos"

    parametros_conexao = urllib.parse.quote_plus(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={servidor};"
        f"DATABASE={base_dados};"
        "Trusted_Connection=yes;"
    )

    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={parametros_conexao}"
    )

    df_final.to_sql(
        name="performance_marketing",
        con=engine,
        if_exists="replace",
        index=False
    )

    print(
        f"{len(df_final)} registos carregados "
        "na tabela performance_marketing."
    )

else:
    print(
        "Carga SQL não executada neste ambiente. "
        "Esta etapa deve ser executada no computador "
        "com SQL Server instalado."
    )

Carga SQL não executada neste ambiente. Esta etapa deve ser executada no computador com SQL Server instalado.


## 10. Conclusões, Limitações e Recomendações

### Principais Resultados

A campanha Ecos Progressivos registou um investimento de **64,79 €**, gerou **25 386 impressões** e alcançou **16 338 pessoas**.

Durante o período analisado, o Google Analytics 4 registou **532 sessões**, **481 novos utilizadores** e **43 inícios de formulário**. O Mailchimp confirmou um total de **16 subscrições**.

A taxa de início do formulário foi de **8,08%**, enquanto a taxa global de conversão por sessão foi de **3,01%**.

O rácio entre as subscrições confirmadas e os inícios de formulário foi de **37,21%**. Este valor deve ser interpretado como uma aproximação operacional, uma vez que compara eventos registados no GA4 com contactos únicos do Mailchimp.

O custo por resultado atribuído pela Meta foi de **9,26 €**. Considerando todas as subscrições registadas no Mailchimp durante o período, o custo global por subscrição foi de **4,05 €**.

### Principais Insights

A maior perda do funil ocorreu antes do início do formulário. Apenas **8,08% das sessões** originaram um evento `form_start`, o que indica que a principal oportunidade de melhoria não está apenas na conclusão do formulário.

A análise sugere que devem ser avaliados elementos como:

- clareza da proposta de valor da newsletter;
- visibilidade do formulário e do botão de subscrição;
- alinhamento entre a mensagem dos anúncios e a landing page;
- experiência em dispositivos móveis;
- qualidade do tráfego enviado para o website.

Entre os utilizadores que iniciaram o formulário, o rácio de subscrição foi de **37,21%**, o que também indica margem para reduzir a fricção durante o processo de registo.

### Limitações da Análise

A análise apresenta algumas limitações:

- os inícios de formulário correspondem a eventos do GA4 e não necessariamente a utilizadores únicos;
- a exportação do Mailchimp não permite identificar diretamente o canal de aquisição de cada subscritor;
- a Meta Ads, o GA4 e o Mailchimp utilizam diferentes métodos de medição e atribuição;
- o custo global por subscrição não deve ser interpretado como CPA exclusivo da campanha paga;
- a dimensão reduzida da campanha limita a generalização dos resultados;
- a extração da tabela de eventos do GA4 depende da posição da secção no ficheiro exportado.

### Recomendações Estratégicas

1. Implementar parâmetros UTM consistentes nos links das campanhas para melhorar a identificação do tráfego pago no GA4.
2. Configurar e validar eventos como `form_start`, `form_submit` e `sign_up`.
3. Melhorar a visibilidade do formulário e reforçar a proposta de valor da newsletter.
4. Avaliar a experiência de subscrição em dispositivos móveis.
5. Testar uma versão simplificada da landing page ou do formulário.
6. Após melhorar a medição, testar Google Search como canal de tráfego baseado em intenção.

### Conclusão Geral

O projeto demonstrou a integração de dados provenientes de Meta Ads, Google Analytics 4 e Mailchimp num pipeline analítico comum.

Os dados foram extraídos e tratados em Python, consolidados num formato vertical, preparados para armazenamento no SQL Server e utilizados na construção de um dashboard em Power BI.

Para além dos resultados da campanha, o projeto permitiu identificar limitações de medição e oportunidades concretas de otimização do funil de aquisição.

A principal conclusão é que o desempenho não depende apenas do volume de tráfego adquirido. A clareza da landing page, a experiência de subscrição e a qualidade da medição também desempenham um papel determinante na conversão.